In [ ]:
"""
==============================================================================
PROJETO CRÉDITO PF (BRASIL) - ETAPA 3: PROJEÇÃO TEMPORAL E SHAP DIRECIONAL
Modelagem Prospectiva (Forward-Looking t+12) com Calibração de Limiar Tau
Atualização: Inclusão de Baseline Naive de Persistência e
Regressão Logística Restrita (Tratamento de Multicolinearidade VIF).
Autor: Itaiguara de Oliveira Bezerra
==============================================================================
"""

import glob
import io
import os
import time
import warnings
import requests
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
import numpy as np
import openpyxl
import pandas as pd
import seaborn as sns

from catboost import CatBoostClassifier, Pool
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, brier_score_loss,
    confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve,
)
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#4a6175"
plt.rcParams["axes.linewidth"] = 0.8
plt.rcParams["grid.color"] = "#dcdcdc"
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["grid.alpha"] = 0.5
pd.set_option("display.float_format", lambda x: "%.4f" % x)

def remover_molduras(ax=None):
    if ax is None: ax = plt.gca()
    for spine in ["top", "right", "left", "bottom"]: ax.spines[spine].set_visible(False)

MODALIDADES_EXPURGO = [
    "Financiamentos à exportação", "Financiamentos à importação",
    "Financiamentos de infraestrutura e desenvolvimento",
    "Financiamentos de títulos e valores mobiliários", "Financiamentos com interveniência",
]

MAPA_NOMES_FORMAIS = {
    "porte": "Faixa de Renda do Tomador", "cnae_ocupacao": "Perfil Ocupacional do Tomador",
    "macrobloco": "Modalidade de Crédito", "log_ticket_medio_rel": "Log Ticket Médio Relativo",
    "spread_bancario": "Spread Bancário", "diff_selic_12m": "Variação Selic Média (12 Meses)",
    "selic_lag3": "Taxa Selic Média (Lag 3m)", "selic_lag6": "Taxa Selic Média (Lag 6m)",
    "diff_endiv_12m": "Variação Endividamento Famílias (12 Meses)", "endiv_lag6": "Endividamento das Famílias (Lag 6m)",
    "crescimento_carteira_interanual": "Crescimento da Carteira (YoY)", "crescimento_operacoes_interanual": "Crescimento de Operações (YoY)",
}

def mapear_macrobloco_pf_vetorizado(df: pd.DataFrame) -> np.ndarray:
    sub = df["submodalidade"].astype(str).str.lower()
    mod = df["modalidade"].astype(str).str.lower()
    cond_rotativo = sub.str.contains("crédito rotativo vinculado a cartão|rotativo|cheque especial|sem consignação|adiantamentos a depositantes|conta garantida|fatura parcelada|saque financiado|antecipação de fatura", regex=True, na=False)
    cond_consignado = sub.str.contains("com consignação|consignado", regex=True, na=False)
    cond_garantia_real = (sub.str.contains("habitacional|sfh|veículos automotores|veículos autom.|home equity|veiculo|veículo|arrendamento financeiro de veículos", regex=True, na=False) | mod.str.contains("imobiliários", regex=True, na=False))
    choices = ["1. Rotativo / Sem Garantia", "2. Consignado / Com Garantia", "3. Garantia Real (Imob/Auto)"]
    conditions = [cond_rotativo, cond_consignado, cond_garantia_real]
    return np.select(conditions, choices, default="4. Outros / Renegociados")

# ------------------------------------------------------------------------------
# 1. LEITURA VIA ARQUIVOS CSV LOCAIS (BKP DO SGS)
# ------------------------------------------------------------------------------
print("=" * 90)
print("--- [1/6] EXTRAÇÃO MACROECONÔMICA LOCAL E ENGENHARIA DE FEATURES ---")
print("=" * 90)

def carregar_csv_local(nome_arquivo, nome_coluna):
    try:
        df = pd.read_csv(f"{nome_arquivo}.csv", sep=";", decimal=",", encoding="latin-1")
    except FileNotFoundError:
        raise RuntimeError(f"Erro: O arquivo '{nome_arquivo}.csv' não foi encontrado.")
    
    meses_pt = {
        'jan': '01', 'fev': '02', 'mar': '03', 'abr': '04', 'mai': '05', 'jun': '06',
        'jul': '07', 'ago': '08', 'set': '09', 'out': '10', 'nov': '11', 'dez': '12'
    }
    
    df.columns = ['data', 'valor']
    df = df[df['data'].astype(str).str.contains('/', na=False)].copy()
    
    df['data'] = df['data'].str.lower()
    for mes_txt, mes_num in meses_pt.items():
        df['data'] = df['data'].str.replace(f"{mes_txt}/", f"{mes_num}/")
        
    def ajustar_ano(data_str):
        partes = str(data_str).split('/')
        if len(partes) == 2 and len(partes[1]) == 2:
            ano = int(partes[1])
            partes[1] = f"19{ano}" if ano > 50 else f"20{ano:02d}"
        return "/".join(partes)
        
    df['data'] = df['data'].apply(ajustar_ano)
    df['data'] = pd.to_datetime(df['data'], format='%m/%Y', errors='coerce')
    
    if df['valor'].dtype == 'O': 
        df['valor'] = df['valor'].str.replace(',', '.')
    
    df[nome_coluna] = pd.to_numeric(df['valor'], errors='coerce')
    df['ano_mes'] = df['data'].dt.to_period('M')
    df_res = df[['ano_mes', nome_coluna]].dropna().drop_duplicates(subset=['ano_mes'])
    print(f"  [OK] Série '{nome_coluna}' carregada via arquivo local ({nome_arquivo}.csv).")
    return df_res

df_selic = carregar_csv_local("selic", "selic_media")
df_juros = carregar_csv_local("juros_pf", "juros_pf")
df_endiv = carregar_csv_local("endividamento_familias", "endividamento_familias")

df_macro = pd.merge(df_selic, df_juros, on="ano_mes", how="inner")
df_macro = pd.merge(df_macro, df_endiv, on="ano_mes", how="inner").sort_values("ano_mes").reset_index(drop=True)
df_macro["spread_bancario"] = df_macro["juros_pf"] - df_macro["selic_media"]
df_macro["diff_selic_12m"] = df_macro["selic_media"].diff(12)
df_macro["diff_endiv_12m"] = df_macro["endividamento_familias"].diff(12)
df_macro["selic_lag3"] = df_macro["selic_media"].shift(3)
df_macro["selic_lag6"] = df_macro["selic_media"].shift(6)
df_macro["endiv_lag6"] = df_macro["endividamento_familias"].shift(6)

# ------------------------------------------------------------------------------
# 2. CARREGAMENTO E MITIGAÇÃO DA FALÁCIA ECOLÓGICA NO TICKET MÉDIO
# ------------------------------------------------------------------------------
print("\n--- [2/6] CARREGANDO E CONSOLIDANDO PAINEL DE CRÉDITO PF ---")
cols_load = ["data_base", "modalidade", "submodalidade", "cnae_ocupacao", "porte", "carteira_ativa", "vencido_acima_de_90_dias", "numero_de_operacoes"]
df_raw = pd.read_parquet("painel_scr_pf_bruto.parquet", columns=cols_load)
df_raw = df_raw[~df_raw["modalidade"].isin(MODALIDADES_EXPURGO)]
df_raw["macrobloco"] = mapear_macrobloco_pf_vetorizado(df_raw)
df_raw["data_base"] = pd.to_datetime(df_raw["data_base"])

df_painel = df_raw.groupby(["data_base", "macrobloco", "cnae_ocupacao", "porte"]).agg(
    carteira_ativa=("carteira_ativa", "sum"), vencido_acima_de_90_dias=("vencido_acima_de_90_dias", "sum"), numero_de_operacoes=("numero_de_operacoes", "sum")
).reset_index()

df_painel["taxa_inadimplencia_pct"] = np.where(df_painel["carteira_ativa"] > 0, (df_painel["vencido_acima_de_90_dias"] / df_painel["carteira_ativa"]) * 100, 0.0)
df_painel["ano_mes"] = df_painel["data_base"].dt.to_period("M")
df_painel = df_painel.sort_values(["macrobloco", "cnae_ocupacao", "porte", "data_base"]).reset_index(drop=True)

ticket_bruto = np.where(df_painel["numero_de_operacoes"] > 0, (df_painel["carteira_ativa"] / df_painel["numero_de_operacoes"]) / 1000.0, 1.0)
df_painel["ticket_medio_mil"] = ticket_bruto
media_mb = df_painel.groupby("macrobloco")["ticket_medio_mil"].transform("mean")
df_painel["log_ticket_medio_rel"] = np.log1p(df_painel["ticket_medio_mil"] / np.maximum(media_mb, 0.01))

cresc_cart = df_painel.groupby(["macrobloco", "cnae_ocupacao", "porte"])["carteira_ativa"].pct_change(12) * 100
cresc_oper = df_painel.groupby(["macrobloco", "cnae_ocupacao", "porte"])["numero_de_operacoes"].pct_change(12) * 100
df_painel["crescimento_carteira_interanual"] = cresc_cart.replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-100, 500)
df_painel["crescimento_operacoes_interanual"] = cresc_oper.replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-100, 500)

HORIZONTE_PROJECAO = 12
df_painel["taxa_inad_futura"] = df_painel.groupby(["macrobloco", "cnae_ocupacao", "porte"])["taxa_inadimplencia_pct"].shift(-HORIZONTE_PROJECAO)

df_final = pd.merge(df_painel, df_macro, on="ano_mes", how="inner").dropna(subset=["taxa_inad_futura"]).reset_index(drop=True)

# ------------------------------------------------------------------------------
# 3. SPLIT TEMPORAL TRIPLO E CALIBRAÇÃO ESTRITA DO LIMIAR TAU
# ------------------------------------------------------------------------------
print("\n--- [3/6] DIVISÃO TEMPORAL TRIPLA E CALIBRAÇÃO ESTRITA DO LIMIAR TAU ---")
corte_val = pd.to_datetime("2020-01-01")
corte_oot = pd.to_datetime("2022-01-01")

df_train_raw = df_final[df_final["data_base"] < corte_val].copy()
df_val_raw = df_final[(df_final["data_base"] >= corte_val) & (df_final["data_base"] < corte_oot)].copy()
df_test_raw = df_final[df_final["data_base"] >= corte_oot].copy()

tau_treino = df_train_raw.groupby("macrobloco").apply(lambda g: (g["vencido_acima_de_90_dias"].sum() / g["carteira_ativa"].sum()) * 100).to_dict()

print("Pontos de Corte Estruturais Calibrados no Treino (< 2020):")
for mb, tau_val in tau_treino.items():
    print(f"  * {mb}: tau = {tau_val:.3f}%")

for df_split in [df_train_raw, df_val_raw, df_test_raw]:
    df_split["tau_treino"] = df_split["macrobloco"].map(tau_treino)
    # DEFINIÇÃO DO ALVO (Conforme Revisão): Y = 1 se Inadimplência em t+12 for maior que o histórico de longo prazo (tau)
    df_split["alvo_inadimplencia_alta"] = (df_split["taxa_inad_futura"] > df_split["tau_treino"]).astype(int)

cat_features = ["macrobloco", "cnae_ocupacao", "porte"]
num_features = ["diff_selic_12m", "selic_lag3", "selic_lag6", "spread_bancario", "diff_endiv_12m", "endiv_lag6", "log_ticket_medio_rel", "crescimento_carteira_interanual", "crescimento_operacoes_interanual"]
num_features_restritas = ["diff_selic_12m", "spread_bancario", "diff_endiv_12m", "log_ticket_medio_rel", "crescimento_carteira_interanual"] 

all_features = cat_features + num_features
all_features_restritas = cat_features + num_features_restritas

X_train, y_train = df_train_raw[all_features], df_train_raw["alvo_inadimplencia_alta"]
X_val, y_val = df_val_raw[all_features], df_val_raw["alvo_inadimplencia_alta"]
X_test, y_test = df_test_raw[all_features], df_test_raw["alvo_inadimplencia_alta"]

df_train_encoded = pd.get_dummies(df_train_raw[all_features], columns=cat_features, dtype=float)
df_val_encoded = pd.get_dummies(df_val_raw[all_features], columns=cat_features, dtype=float)
df_test_encoded = pd.get_dummies(df_test_raw[all_features], columns=cat_features, dtype=float)
df_train_encoded, df_val_encoded = df_train_encoded.align(df_val_encoded, join="left", axis=1, fill_value=0.0)
df_train_encoded, df_test_encoded = df_train_encoded.align(df_test_encoded, join="left", axis=1, fill_value=0.0)

df_train_encoded_restrita = pd.get_dummies(df_train_raw[all_features_restritas], columns=cat_features, dtype=float)
df_test_encoded_restrita = pd.get_dummies(df_test_raw[all_features_restritas], columns=cat_features, dtype=float)
df_train_encoded_restrita, df_test_encoded_restrita = df_train_encoded_restrita.align(df_test_encoded_restrita, join="left", axis=1, fill_value=0.0)

# ------------------------------------------------------------------------------
# 4. TREINAMENTO DOS MODELOS, BASELINE NAIVE E AVALIAÇÃO OOT
# ------------------------------------------------------------------------------
print("\n--- [4/6] TREINAMENTO DOS MODELOS E AVALIAÇÃO OOT (PROJEÇÃO t+12 MESES) ---")

# 0. Baseline Naive (Persistência)
y_prob_naive = (df_test_raw["taxa_inadimplencia_pct"] > df_test_raw["tau_treino"]).astype(float)
y_pred_naive = (y_prob_naive > 0.5).astype(int)

# 1. Regressão Logística (Completa)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(df_train_encoded)
X_te_sc = scaler.transform(df_test_encoded)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_tr_sc, y_train)
y_pred_lr = lr.predict(X_te_sc)
y_prob_lr = lr.predict_proba(X_te_sc)[:, 1]

# 1.5 Regressão Logística (Restrita - Tratamento Multicolinearidade)
scaler_restrita = StandardScaler()
X_tr_sc_res = scaler_restrita.fit_transform(df_train_encoded_restrita)
X_te_sc_res = scaler_restrita.transform(df_test_encoded_restrita)
lr_res = LogisticRegression(max_iter=1000, random_state=42)
lr_res.fit(X_tr_sc_res, y_train)
y_pred_lr_res = lr_res.predict(X_te_sc_res)
y_prob_lr_res = lr_res.predict_proba(X_te_sc_res)[:, 1]

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf.fit(df_train_encoded, y_train)
y_pred_rf = rf.predict(df_test_encoded)
y_prob_rf = rf.predict_proba(df_test_encoded)[:, 1]

# 3. CatBoost Classifier com Calibração Isotônica
pool_train = Pool(X_train, y_train, cat_features=cat_features)
pool_val = Pool(X_val, y_val, cat_features=cat_features)
pool_test = Pool(X_test, y_test, cat_features=cat_features)

cb_base = CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6, random_seed=42, verbose=False)
cb_base.fit(pool_train, eval_set=pool_val, early_stopping_rounds=40, verbose=False)
calibrador = CalibratedClassifierCV(cb_base, cv="prefit", method="isotonic")
calibrador.fit(X_val, y_val)
y_pred_cb = calibrador.predict(X_test)
y_prob_cb = calibrador.predict_proba(X_test)[:, 1]

# ------------------------------------------------------------------------------
# 5. QUADRO COMPARATIVO CONSOLIDADO
# ------------------------------------------------------------------------------
modelos_dict = {
    "0. Baseline Naive (Persistência)": (y_pred_naive, y_prob_naive),
    "1. Regressão Logística (Completa)": (y_pred_lr, y_prob_lr),
    "2. Regressão Logística (Restrita)": (y_pred_lr_res, y_prob_lr_res),
    "3. Random Forest": (y_pred_rf, y_prob_rf),
    "4. CatBoost Classifier (Calibrado)": (y_pred_cb, y_prob_cb),
}

tabela_metricas = []
for nome, (pred, prob) in modelos_dict.items():
    tabela_metricas.append({
        "Modelo": nome,
        "Acurácia": f"{accuracy_score(y_test, pred):.2%}",
        "ROC-AUC": f"{roc_auc_score(y_test, prob):.4f}",
        "PR-AUC (Avg Precision)": f"{average_precision_score(y_test, prob):.4f}",
        "Brier Score": f"{brier_score_loss(y_test, prob):.4f}",
    })

df_resultado_oot = pd.DataFrame(tabela_metricas)

print("\n" + "=" * 90)
print("ETAPA 3: RESULTADO OFICIAL - VALIDAÇÃO OUT-OF-TIME CRÉDITO PF (2022-2026)")
print("PROJEÇÃO FORWARD-LOOKING (t + 12 MESES) COM TESTE DE PERSISTÊNCIA")
print("=" * 90)
print(df_resultado_oot.to_string(index=False))
print("=" * 90)

nome_excel = "relatorio_etapa3_pf_modelagem.xlsx"
with pd.ExcelWriter(nome_excel, engine="openpyxl") as writer:
    df_resultado_oot.to_excel(writer, sheet_name="Metricas_Comparativas_OOT", index=False)
    pd.DataFrame(list(tau_treino.items()), columns=["Modalidade de Crédito", "Tau_Treino_Pct"]).to_excel(writer, sheet_name="Parametros_Tau", index=False)

# ------------------------------------------------------------------------------
# 6. GERAÇÃO COMPLETA DE TODOS OS GRÁFICOS (ROC, PR E MATRIZ DE CONFUSÃO)
# ------------------------------------------------------------------------------
print("\n--- [5/6] GERANDO MATRIZES DE CONFUSÃO E CURVAS DE PERFORMANCE ---")

cores_modelos = ["#7f7f7f", "#9467bd", "#1f77b4", "#ff7f0e", "#2ca02c"]

# 1. Matrizes de Confusão 2x2 (Selecionamos os 4 principais para o grid)
fig_cm, axes_cm = plt.subplots(2, 2, figsize=(11, 9), dpi=300)
plots_config = [
    (y_pred_naive, "Greys", axes_cm[0, 0], "Baseline Naive (Persistência)"),
    (y_pred_lr, "Purples", axes_cm[0, 1], "Regressão Logística (Completa)"),
    (y_pred_rf, "Oranges", axes_cm[1, 0], "Random Forest"),
    (y_pred_cb, "Greens", axes_cm[1, 1], "CatBoost Classifier (Calibrado)"),
]

for pred, cmap, ax, titulo_sub in plots_config:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, ax=ax, cbar=False, annot_kws={"size": 12, "weight": "bold"})
    ax.set_title(titulo_sub, fontsize=11, fontweight="bold", pad=10, color="#16222f")
    ax.set_xlabel("Previsto", fontweight="bold", fontsize=9.5)
    ax.set_ylabel("Real", fontweight="bold", fontsize=9.5)
    ax.set_xticklabels(["Baixo (0)", "Alto (1)"])
    ax.set_yticklabels(["Baixo (0)", "Alto (1)"])

fig_cm.tight_layout()
plt.savefig("etapa3_pf_matrizes_confusao_oot.png", dpi=300, bbox_inches="tight")
plt.close()
print("  [OK] Imagem salva: 'etapa3_pf_matrizes_confusao_oot.png'")

# 2. Curva ROC (Todos os 5 Modelos)
plt.figure(figsize=(9, 6.5), dpi=300)
for (nome, (_, prob)), cor in zip(modelos_dict.items(), cores_modelos):
    fpr, tpr, _ = roc_curve(y_test, prob)
    auc_val = roc_auc_score(y_test, prob)
    plt.plot(fpr, tpr, label=f"{nome} (AUC = {auc_val:.4f})", color=cor, lw=2.2)

plt.plot([0, 1], [0, 1], color="gray", linestyle=":", lw=1.5, label="Classificador Aleatório")
plt.xlabel("Taxa de Falsos Positivos (1 - Especificidade)", fontsize=10, fontweight="bold")
plt.ylabel("Taxa de Verdadeiros Positivos (Sensibilidade)", fontsize=10, fontweight="bold")
plt.legend(loc="lower right", frameon=False, fontsize=9.0)
remover_molduras()
plt.tight_layout()
plt.savefig("etapa3_pf_curva_roc_oot.png", dpi=300, bbox_inches="tight")
plt.close()
print("  [OK] Imagem salva: 'etapa3_pf_curva_roc_oot.png'")

# 3. Curva Precision-Recall (PR)
taxa_prevalencia_teste = y_test.mean()
plt.figure(figsize=(9, 6.5), dpi=300)
for (nome, (_, prob)), cor in zip(modelos_dict.items(), cores_modelos):
    precision, recall, _ = precision_recall_curve(y_test, prob)
    pr_auc_val = average_precision_score(y_test, prob)
    plt.plot(recall, precision, label=f"{nome} (PR-AUC = {pr_auc_val:.4f})", color=cor, lw=2.2)

plt.axhline(y=taxa_prevalencia_teste, color="gray", linestyle=":", lw=1.5, label=f"Aleatório ({taxa_prevalencia_teste:.2%})")
plt.xlabel("Recall (Sensibilidade)", fontsize=10, fontweight="bold")
plt.ylabel("Precisão (Valor Preditivo Positivo)", fontsize=10, fontweight="bold")
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.legend(loc="best", frameon=False, fontsize=9.0)
remover_molduras()
plt.tight_layout()
plt.savefig("etapa3_pf_curva_pr_oot.png", dpi=300, bbox_inches="tight")
plt.close()
print("  [OK] Imagem salva: 'etapa3_pf_curva_pr_oot.png'")

# ------------------------------------------------------------------------------
# 7. EXPLICABILIDADE SHAP: RANKING DIRECIONAL E HEATMAP POR MACROBLOCO
# ------------------------------------------------------------------------------
print("\n--- [6/6] CALCULANDO VALORES SHAP DIRECIONAIS (CATBOOST OOT) ---")

shap_matrix = cb_base.get_feature_importance(pool_test, type="ShapValues")
shap_values = shap_matrix[:, :-1]
df_shap = pd.DataFrame(shap_values, columns=all_features)

feature_importance = []
for feat in all_features:
    mean_abs_shap = np.mean(np.abs(df_shap[feat]))
    nome_formal = MAPA_NOMES_FORMAIS.get(feat, feat)

    if feat in num_features:
        corr = np.corrcoef(X_test[feat].values, df_shap[feat].values)[0, 1]
        direcao = "Eleva Risco (+)" if corr > 0 else "Fator Protetivo (-)"
        cor_hex = "#d62728" if corr > 0 else "#1f77b4"
    else:
        direcao = "Segmentação Estrutural"
        cor_hex = "#4a6175"
        corr = np.nan

    feature_importance.append({
        "Feature_Cod": feat,
        "Variável (Opção Formal)": nome_formal,
        "Mean_Abs_SHAP": mean_abs_shap,
        "Direção": direcao,
        "Cor": cor_hex,
        "Correlação_com_Risco": corr,
    })

df_importance = pd.DataFrame(feature_importance).sort_values("Mean_Abs_SHAP", ascending=False).reset_index(drop=True)

print("\n--- RANKING DE IMPORTÂNCIA GLOBAL SHAP (NOMES FORMAIS) ---")
print(df_importance[["Variável (Opção Formal)", "Mean_Abs_SHAP", "Direção"]].to_string(index=False))

# 4. Gráfico de Barras Horizontais SHAP Direcional
plt.figure(figsize=(10.5, 7.5), dpi=300)
df_plot = df_importance.sort_values("Mean_Abs_SHAP", ascending=True)

bars = plt.barh(df_plot["Variável (Opção Formal)"], df_plot["Mean_Abs_SHAP"], color=df_plot["Cor"], edgecolor="none", height=0.65)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.005, bar.get_y() + bar.get_height() / 2, f"{w:.3f}", va="center", ha="left", fontsize=9, fontweight="bold", color="#2c3e50")

plt.xlabel("Impacto Médio no Log-Odds de Inadimplência (|SHAP Value|)", fontsize=10, fontweight="bold")
plt.xlim(0, df_plot["Mean_Abs_SHAP"].max() * 1.20)

legend_elements = [
    Patch(facecolor="#d62728", label="Eleva Risco de Inadimplência (+)"),
    Patch(facecolor="#1f77b4", label="Fator Protetivo / Mitigador (-)"),
    Patch(facecolor="#4a6175", label="Segmentação Estrutural (Renda / Ocupação / Linha)"),
]
plt.legend(handles=legend_elements, loc="lower right", frameon=False, fontsize=9)
remover_molduras()
plt.tight_layout()
plt.savefig("etapa3_pf_shap_importancia_direcional.png", dpi=300, bbox_inches="tight")
plt.close()
print("  [OK] Imagem salva: 'etapa3_pf_shap_importancia_direcional.png'")

# 5. Heatmap SHAP por Macrobloco de Crédito (Com Limpeza de Numerais e Fonte Otimizada)
X_test_mb = X_test.copy().reset_index(drop=True)
df_shap["macrobloco"] = X_test_mb["macrobloco"]
shap_por_mb = df_shap.groupby("macrobloco")[num_features].apply(lambda g: np.mean(np.abs(g), axis=0))
shap_por_mb.columns = [MAPA_NOMES_FORMAIS.get(col, col) for col in shap_por_mb.columns]

# Remove os numerais (ex: "1. ", "2. ") do nome das modalidades para o eixo X
shap_por_mb.index = shap_por_mb.index.str.replace(r'^\d+\.\s*', '', regex=True)

plt.figure(figsize=(11.5, 7.5), dpi=300)
sns.heatmap(shap_por_mb.T, annot=True, fmt=".3f", cmap="YlOrRd", linewidths=1.0, linecolor="#ffffff", cbar=False, annot_kws={"size": 8.5, "weight": "bold"})
plt.xlabel("Modalidade de Crédito", fontweight="bold", fontsize=10)
plt.ylabel("Variáveis Macroeconômicas e Contratuais", fontweight="bold", fontsize=10)
plt.xticks(fontsize=8.5)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig("etapa3_pf_shap_por_macrobloco_heatmap.png", dpi=300, bbox_inches="tight")
plt.close()
print("  [OK] Imagem salva: 'etapa3_pf_shap_por_macrobloco_heatmap.png'")

# Salva abas no Excel
with pd.ExcelWriter(nome_excel, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    df_importance.to_excel(writer, sheet_name="SHAP_Global_Ranking", index=False)
    shap_por_mb.to_excel(writer, sheet_name="SHAP_Por_Modalidade")

print(f"\n[OK] Métricas e SHAP salvos na planilha: '{nome_excel}'")
print("\n" + "=" * 90)
print("--- [ETAPA 3 CONCLUÍDA COM RIGOR ECONOMÉTRICO TOTAL!] ---")
print("=" * 90)